In [ ]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

import datachain as dc 
from datachain.func.path import file_ext, file_stem, name, parent
from transformers import Pipeline, pipeline
from datachain import File

from FantAIno.constants import S3_GENERAL_PURPOSE_BUCKET_NAME
from FantAIno.utils.data_utils import get_secret

# Loading in a Multi-Modal Dataset from S3 with DataChain

In [ ]:
image_df = (dc.read_storage(
    rf"s3://{S3_GENERAL_PURPOSE_BUCKET_NAME}/album_art\*",
    type="image", 
    client_config = {
        "key": get_secret("AWS_ACCESS_KEY_ID"),
        "secret": get_secret("AWS_SECRET_ACCESS_KEY")
    }
)
    .limit(100)
    .settings(cache=True)
    .map(path=lambda file: file.path, output=str)
    .persist()
)

In [4]:
image_df.show(3)

,file,file,path
,path,size,
0,album_art\$NOT___Ethereal.jpg,27229,album_art\$NOT___Ethereal.jpg
1,album_art\$uicideboy$___I Want to Die in New O...,38617,album_art\$uicideboy$___I Want to Die in New O...
2,album_art\$uicideboy$___Long Term Effects of S...,443669,album_art\$uicideboy$___Long Term Effects of S...



[Limited by 3 rows]


In [ ]:
lyrics_df = (dc.read_storage(
    rf"s3://{S3_GENERAL_PURPOSE_BUCKET_NAME}/lyrics\*",
    client_config = {
        "key": get_secret("AWS_ACCESS_KEY_ID"),
        "secret": get_secret("AWS_SECRET_ACCESS_KEY")
    }
)
    .limit(100)
    .settings(cache=True)
    .map(path=lambda file: file.path, output=str)
    .persist()
)

In [6]:
lyrics_df.show(3)

,file,file,path
,path,size,
0,lyrics\$NOT___Ethereal.jsonl,32775,lyrics\$NOT___Ethereal.jsonl
1,lyrics\$uicideboy$___I Want to Die in New Orle...,36797,lyrics\$uicideboy$___I Want to Die in New Orle...
2,lyrics\$uicideboy$___Long Term Effects of SUFF...,31488,lyrics\$uicideboy$___Long Term Effects of SUFF...



[Limited by 3 rows]


In [19]:
dc_catalog = (
    image_df
    .settings(cache=True)
    .merge(lyrics_df, on=name(file_stem((image_df.c("file.path")))))
    .persist()
)

In [21]:
dc_catalog.show(include_hidden=True)

,file,file,file,file,file,file,file,file,path,right_file,right_file,right_file,right_file,right_file,right_file,right_file,right_file,right_path
,source,path,size,version,etag,is_latest,last_modified,location,,source,path,size,version,etag,is_latest,last_modified,location,
0,s3://fantaino-bucket-085777795487-us-east-2-an,album_art\$NOT___Ethereal.jpg,27229,cWCWit8Zp.CCGEusjTKrqkdqaW_OGEKr,addc3053ccaf2f6826ad906b9f8bd5fe,True,2026-05-02 08:08:28+00:00,None,album_art\$NOT___Ethereal.jpg,s3://fantaino-bucket-085777795487-us-east-2-an,lyrics\$NOT___Ethereal.jsonl,32775,5VMyF0k7SprY6E0MhcIwPtHEXNXf..Dn,ae2f5afc656b2754dfb3214ec539240a,True,2026-05-02 08:08:46+00:00,None,lyrics\$NOT___Ethereal.jsonl
1,s3://fantaino-bucket-085777795487-us-east-2-an,album_art\$NOT___Ethereal.jpg,27229,cWCWit8Zp.CCGEusjTKrqkdqaW_OGEKr,addc3053ccaf2f6826ad906b9f8bd5fe,True,2026-05-02 08:08:28+00:00,None,album_art\$NOT___Ethereal.jpg,s3://fantaino-bucket-085777795487-us-east-2-an,lyrics\$uicideboy$___I Want to Die in New Orle...,36797,oqYTUds011b5_rYuF8YENrmEodBS975F,0c01434f6c3cf2142cf27875ea4f460f,True,2026-05-02 17:35:54+00:00,None,lyrics\$uicideboy$___I Want to Die in New Orle...
2,s3://fantaino-bucket-085777795487-us-east-2-an,album_art\$NOT___Ethereal.jpg,27229,cWCWit8Zp.CCGEusjTKrqkdqaW_OGEKr,addc3053ccaf2f6826ad906b9f8bd5fe,True,2026-05-02 08:08:28+00:00,None,album_art\$NOT___Ethereal.jpg,s3://fantaino-bucket-085777795487-us-east-2-an,lyrics\$uicideboy$___Long Term Effects of SUFF...,31488,7Md9AIbQ6z_neYXoive8wMX4U3W6j0fL,909033741937a5ccb4fe64f4456ec495,True,2026-05-02 09:32:12+00:00,None,lyrics\$uicideboy$___Long Term Effects of SUFF...
3,s3://fantaino-bucket-085777795487-us-east-2-an,album_art\$NOT___Ethereal.jpg,27229,cWCWit8Zp.CCGEusjTKrqkdqaW_OGEKr,addc3053ccaf2f6826ad906b9f8bd5fe,True,2026-05-02 08:08:28+00:00,None,album_art\$NOT___Ethereal.jpg,s3://fantaino-bucket-085777795487-us-east-2-an,lyrics\$uicideboy$___New World Depression.jsonl,39757,28Yc7TInfQjQP01hB9P5_UIQzXa6xEFy,c11c4b87e3ba7f944e4b8b3df43ce652,True,2026-05-02 01:06:22+00:00,None,lyrics\$uicideboy$___New World Depression.jsonl
4,s3://fantaino-bucket-085777795487-us-east-2-an,album_art\$NOT___Ethereal.jpg,27229,cWCWit8Zp.CCGEusjTKrqkdqaW_OGEKr,addc3053ccaf2f6826ad906b9f8bd5fe,True,2026-05-02 08:08:28+00:00,None,album_art\$NOT___Ethereal.jpg,s3://fantaino-bucket-085777795487-us-east-2-an,lyrics\070 Shake___You Cant Kill Me.jsonl,24121,p6lFidSafJCAc0yHXwPhyPH7BFn4zt4p,130b8f7d083cea7f4162965e8c403f9e,True,2026-05-02 07:16:23+00:00,None,lyrics\070 Shake___You Cant Kill Me.jsonl
5,s3://fantaino-bucket-085777795487-us-east-2-an,album_art\$NOT___Ethereal.jpg,27229,cWCWit8Zp.CCGEusjTKrqkdqaW_OGEKr,addc3053ccaf2f6826ad906b9f8bd5fe,True,2026-05-02 08:08:28+00:00,None,album_art\$NOT___Ethereal.jpg,s3://fantaino-bucket-085777795487-us-east-2-an,"lyrics\100 gecs___10,000 gecs.jsonl",20770,l.Lw2sLLiS2VkekFR.KJE_BZmjXr.DdL,b368e7d821f9e8b4dbf6341be01c8438,True,2026-05-02 05:09:23+00:00,None,"lyrics\100 gecs___10,000 gecs.jsonl"
6,s3://fantaino-bucket-085777795487-us-east-2-an,album_art\$NOT___Ethereal.jpg,27229,cWCWit8Zp.CCGEusjTKrqkdqaW_OGEKr,addc3053ccaf2f6826ad906b9f8bd5fe,True,2026-05-02 08:08:28+00:00,None,album_art\$NOT___Ethereal.jpg,s3://fantaino-bucket-085777795487-us-east-2-an,lyrics\100 gecs___1000 gecs and The Tree of Cl...,40381,5cS8dIhXOmZE0OHlTSRgRQQdlTIp6fey,55d0a58d410e635b01ab4b876c8e2e26,True,2026-05-02 12:25:00+00:00,None,lyrics\100 gecs___1000 gecs and The Tree of Cl...
7,s3://fantaino-bucket-085777795487-us-east-2-an,album_art\$NOT___Ethereal.jpg,27229,cWCWit8Zp.CCGEusjTKrqkdqaW_OGEKr,addc3053ccaf2f6826ad906b9f8bd5fe,True,2026-05-02 08:08:28+00:00,None,album_art\$NOT___Ethereal.jpg,s3://fantaino-bucket-085777795487-us-east-2-an,lyrics\100 gecs___1000 gecs.jsonl,16731,j_yxrTYwF4lHP0N29NI8ebLRQR.AaQ0C,73e95531b89a99d4b198aafea1bb6157,True,2026-05-02 15:33:48+00:00,None,lyrics\100 gecs___1000 gecs.jsonl
8,s3://fantaino-bucket-085777795487-us-east-2-an,album_art\$NOT___Et


[Limited by 20 rows]


In [24]:
dc_catalog.show()

,file,file,path,right_file,right_file,right_path
,path,size,,path,size,
0,album_art\$NOT___Ethereal.jpg,27229,album_art\$NOT___Ethereal.jpg,lyrics\$NOT___Ethereal.jsonl,32775,lyrics\$NOT___Ethereal.jsonl
1,album_art\$NOT___Ethereal.jpg,27229,album_art\$NOT___Ethereal.jpg,lyrics\$uicideboy$___I Want to Die in New Orle...,36797,lyrics\$uicideboy$___I Want to Die in New Orle...
2,album_art\$NOT___Ethereal.jpg,27229,album_art\$NOT___Ethereal.jpg,lyrics\$uicideboy$___Long Term Effects of SUFF...,31488,lyrics\$uicideboy$___Long Term Effects of SUFF...
3,album_art\$NOT___Ethereal.jpg,27229,album_art\$NOT___Ethereal.jpg,lyrics\$uicideboy$___New World Depression.jsonl,39757,lyrics\$uicideboy$___New World Depression.jsonl
4,album_art\$NOT___Ethereal.jpg,27229,album_art\$NOT___Ethereal.jpg,lyrics\070 Shake___You Cant Kill Me.jsonl,24121,lyrics\070 Shake___You Cant Kill Me.jsonl
5,album_art\$NOT___Ethereal.jpg,27229,album_art\$NOT___Ethereal.jpg,"lyrics\100 gecs___10,000 gecs.jsonl",20770,"lyrics\100 gecs___10,000 gecs.jsonl"
6,album_art\$NOT___Ethereal.jpg,27229,album_art\$NOT___Ethereal.jpg,lyrics\100 gecs___1000 gecs and The Tree of Cl...,40381,lyrics\100 gecs___1000 gecs and The Tree of Cl...
7,album_art\$NOT___Ethereal.jpg,27229,album_art\$NOT___Ethereal.jpg,lyrics\100 gecs___1000 gecs.jsonl,16731,lyrics\100 gecs___1000 gecs.jsonl
8,album_art\$NOT___Ethereal.jpg,27229,album_art\$NOT___Ethereal.jpg,lyrics\2 Chainz___Rap or Go to the League.jsonl,44643,lyrics\2 Chainz___Rap or Go to the League.jsonl



[Limited by 20 rows]


In [ ]:
import matplotlib.pyplot as plt
from textwrap import wrap

count = chain.count()
_, axes = plt.subplots(1, count, figsize=(15, 5))

for ax, (img_file, caption) in zip(axes, chain.to_iter("file", "scene")):
    ax.imshow(img_file.read(), cmap="gray")
    ax.axis("off")
    wrapped_caption = "\n".join(wrap(caption.strip(), 40))
    ax.set_title(wrapped_caption, fontsize=10, pad=20)

plt.tight_layout()
plt.show()

# Loading from S3 Table Buckets

In [ ]:
embeddings_df = (dc.read_storage(
    r"s3://fantaino-bucket-085777795487-us-east-2-an/lyrics\*",
    client_config = {
        "key": get_secret("AWS_ACCESS_KEY_ID"),
        "secret": get_secret("AWS_SECRET_ACCESS_KEY")
    }
)
    .limit(100)
    .settings(cache=True)
    .map(path=lambda file: file.path, output=str)
    .persist()
)

# Creating Multi-Modal Datasets